In [1]:
# cell 1
# Install runtime dependencies.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy accelerate safetensors huggingface_hub

# Remove optional packages before installing vLLM.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Gemma 4 needs recent Transformers support.
!uv pip install --system -U "transformers>=5.5.0"

# Recent vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Keep the Prometheus instrumentator on a stable 7.x version.
!uv pip install --system -U "prometheus-fastapi-instrumentator==7.1.0"

# Remove only packages that are not needed after vLLM installation.
# Do NOT uninstall torchvision or torchaudio here.
!uv pip uninstall --system -y torchcodec sentence-transformers || true

# Patch prometheus-fastapi-instrumentator for newer FastAPI/Starlette router objects.
# This avoids:
# AttributeError: '_IncludedRouter' object has no attribute 'path'
from pathlib import Path
import importlib.util
import shutil

def patch_prometheus_fastapi_instrumentator_routing():
    # Patch the installed package source so the vLLM subprocess also sees it.
    spec = importlib.util.find_spec("prometheus_fastapi_instrumentator.routing")

    if spec is None or spec.origin is None:
        print("prometheus_fastapi_instrumentator.routing was not found; skipping patch.")
        return

    routing_path = Path(spec.origin)
    text = routing_path.read_text(encoding="utf-8")

    marker = "# --- vLLM Starlette _IncludedRouter compatibility patch ---"

    if marker in text:
        print("Prometheus routing patch already applied:", routing_path)
        return

    backup_path = routing_path.with_suffix(routing_path.suffix + ".bak")

    if not backup_path.exists():
        shutil.copy2(routing_path, backup_path)

    patch = f'''

{marker}
def get_route_name(request):
    """
    Compatibility override for newer FastAPI/Starlette route objects.

    Some Starlette versions expose internal router objects without a `.path`
    attribute. prometheus-fastapi-instrumentator may crash while trying to
    inspect those objects. For vLLM serving, using the raw request path as the
    metrics route name is sufficient and prevents API requests from failing.
    """
    try:
        scope = getattr(request, "scope", None) or {{}}
        route = scope.get("route")
        route_path = getattr(route, "path", None)

        if route_path:
            return route_path

        return scope.get("path") or "__unknown__"

    except Exception:
        return "__unknown__"
# --- end vLLM compatibility patch ---
'''

    routing_path.write_text(text + patch, encoding="utf-8")

    # Remove stale pyc files so the subprocess imports the patched source.
    try:
        for pycache in routing_path.parent.rglob("__pycache__"):
            for pyc in pycache.glob("routing*.pyc"):
                pyc.unlink()
    except Exception as e:
        print("Warning: could not remove routing pyc files:", repr(e))

    print("Applied Prometheus routing patch:", routing_path)
    print("Backup saved at:", backup_path)

patch_prometheus_fastapi_instrumentator_routing()

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in [
    "prometheus-fastapi-instrumentator",
    "fastapi",
    "starlette",
    "torchcodec",
    "torchvision",
    "torchaudio",
    "sentence-transformers",
]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 112ms
Prepared 8 packages in 0.33ms
Uninstalled 8 packages in 124ms
Installed 8 packages in 109ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 49ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 70ms
Checked 27 packages in 0.32ms
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 7.97s
Prepared 12 packages in 13ms
Uninstalled 10 packages in 109ms
Installed 12 packages in 112ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cublas

In [2]:
# cell 2
# Imports and global config.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback

from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI
from jsonschema import validate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

LLM_MODEL_NAME = "google/gemma-4-26B-A4B-it"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Gemma 4 26B-A4B is documented in vLLM recipes with 131072 context.
MAX_MODEL_LEN = 131072

GPU_MEMORY_UTILIZATION = 0.92

MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

SERVER_LOG_PATH = Path("/content/vllm_gemma4_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gemma4_answer_server.pid")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
DRIVE_EVIDENCE_DIR = DRIVE_PROJECT_DIR / "RAG" / "evidence"
DRIVE_LLM_DIR = DRIVE_PROJECT_DIR / "LLM"

LOCAL_RUNTIME_DIR = Path("/content/final_project_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "RAG" / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = [
    {
        "name": "2wikimultihopqa",
        "evidence_filename": "2wikimultihopqa_evidence.json",
        "output_filename": "2wikimultihopqa_gemma4_answers.json",
    },
    {
        "name": "hotpotqa",
        "evidence_filename": "hotpotqa_evidence.json",
        "output_filename": "hotpotqa_gemma4_answers.json",
    },
]

for cfg in DATASET_CONFIGS:
    cfg["drive_evidence_path"] = DRIVE_EVIDENCE_DIR / cfg["evidence_filename"]
    cfg["local_evidence_path"] = LOCAL_EVIDENCE_DIR / cfg["evidence_filename"]
    cfg["drive_output_path"] = DRIVE_LLM_DIR / cfg["output_filename"]

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None  # None means all records for each dataset.

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# Non-thinking short-answer budget.
# The prompt asks for less than 6 words, so 64 is enough and much faster.
ANSWER_MAX_TOKENS = 64

LOW_COMPLETION_TOKENS_THRESHOLD = 16

print("Model:", LLM_MODEL_NAME)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Answer max tokens:", ANSWER_MAX_TOKENS)
print("Drive evidence directory:", DRIVE_EVIDENCE_DIR)
print("Drive LLM output directory:", DRIVE_LLM_DIR)

for cfg in DATASET_CONFIGS:
    print("=" * 100)
    print("Dataset:", cfg["name"])
    print("Drive evidence path:", cfg["drive_evidence_path"])
    print("Local evidence path:", cfg["local_evidence_path"])
    print("Output path:", cfg["drive_output_path"])

Model: google/gemma-4-26B-A4B-it
Max model len: 131072
GPU memory utilization: 0.92
Answer max tokens: 64
Drive evidence directory: /content/drive/MyDrive/final_project/RAG/evidence
Drive LLM output directory: /content/drive/MyDrive/final_project/LLM
Dataset: 2wikimultihopqa
Drive evidence path: /content/drive/MyDrive/final_project/RAG/evidence/2wikimultihopqa_evidence.json
Local evidence path: /content/final_project_copy/RAG/evidence/2wikimultihopqa_evidence.json
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json
Dataset: hotpotqa
Drive evidence path: /content/drive/MyDrive/final_project/RAG/evidence/hotpotqa_evidence.json
Local evidence path: /content/final_project_copy/RAG/evidence/hotpotqa_evidence.json
Output path: /content/drive/MyDrive/final_project/LLM/hotpotqa_gemma4_answers.json


In [3]:
# cell 3
# Mount Google Drive and copy evidence files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

DRIVE_LLM_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether local copy is complete.
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with a temp file to avoid partial local copies.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print("Local evidence copy already exists:", dst)
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

for cfg in DATASET_CONFIGS:
    src = cfg["drive_evidence_path"]
    dst = cfg["local_evidence_path"]

    if not src.exists():
        raise FileNotFoundError(f"Evidence file not found: {src}")

    copy_file_to_local(src, dst)

    print("=" * 100)
    print("Dataset:", cfg["name"])
    print("Evidence copied to local disk.")
    print("Local evidence size MB:", dst.stat().st_size / (1024 ** 2))
    print("Output file will be:", cfg["drive_output_path"])

Mounted at /content/drive
Local evidence copy already exists: /content/final_project_copy/RAG/evidence/2wikimultihopqa_evidence.json
Dataset: 2wikimultihopqa
Evidence copied to local disk.
Local evidence size MB: 5.524371147155762
Output file will be: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json
Local evidence copy already exists: /content/final_project_copy/RAG/evidence/hotpotqa_evidence.json
Dataset: hotpotqa
Evidence copied to local disk.
Local evidence size MB: 6.91798210144043
Output file will be: /content/drive/MyDrive/final_project/LLM/hotpotqa_gemma4_answers.json


In [4]:
# cell 4
# Load and validate both evidence datasets.

dataset_records = {}

required_keys = ["type", "question", "answer"]

for cfg in DATASET_CONFIGS:
    dataset_name = cfg["name"]
    local_path = cfg["local_evidence_path"]

    with open(local_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise RuntimeError(f"Evidence JSON must be a list of records: {local_path}")

    for i, rec in enumerate(records[:5]):
        missing = [k for k in required_keys if k not in rec]
        if missing:
            print(f"Warning: dataset {dataset_name}, record {i} missing keys:", missing)

    dataset_records[dataset_name] = records

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Number of evidence records:", len(records))

    if records:
        print("First question:", records[0].get("question"))
        print("First GT answer:", records[0].get("answer"))
        print("First type:", records[0].get("type"))
        print("Has supports:", "supports" in records[0])
        print("Has evidence_chunk:", "evidence_chunk" in records[0])

Dataset: 2wikimultihopqa
Number of evidence records: 1000
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT answer: Kamakalawa
First type: bridge_comparison
Has supports: True
Has evidence_chunk: True
Dataset: hotpotqa
Number of evidence records: 1000
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT answer: Bedknobs and Broomsticks
First type: comparison
Has supports: True
Has evidence_chunk: True


In [5]:
# cell 5
# Start Gemma 4 vLLM server in non-thinking mode.

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def patch_prometheus_fastapi_instrumentator_routing_again():
    # Re-apply the routing patch before starting the vLLM subprocess.
    try:
        import importlib.util
        import shutil

        spec = importlib.util.find_spec("prometheus_fastapi_instrumentator.routing")

        if spec is None or spec.origin is None:
            print("prometheus_fastapi_instrumentator.routing was not found; skipping patch.")
            return

        routing_path = Path(spec.origin)
        text = routing_path.read_text(encoding="utf-8")

        marker = "# --- vLLM Starlette _IncludedRouter compatibility patch ---"

        if marker in text:
            print("Prometheus routing patch already applied:", routing_path)
            return

        backup_path = routing_path.with_suffix(routing_path.suffix + ".bak")

        if not backup_path.exists():
            shutil.copy2(routing_path, backup_path)

        patch = f'''

{marker}
def get_route_name(request):
    """
    Compatibility override for newer FastAPI/Starlette route objects.
    """
    try:
        scope = getattr(request, "scope", None) or {{}}
        route = scope.get("route")
        route_path = getattr(route, "path", None)

        if route_path:
            return route_path

        return scope.get("path") or "__unknown__"

    except Exception:
        return "__unknown__"
# --- end vLLM compatibility patch ---
'''

        routing_path.write_text(text + patch, encoding="utf-8")

        try:
            for pycache in routing_path.parent.rglob("__pycache__"):
                for pyc in pycache.glob("routing*.pyc"):
                    pyc.unlink()
        except Exception:
            pass

        print("Applied Prometheus routing patch:", routing_path)

    except Exception as e:
        print("Warning: could not patch Prometheus routing:", repr(e))

def find_or_download_gemma4_chat_template():
    # Find or download the Gemma 4 vLLM chat template.
    template_name = "tool_chat_template_gemma4.jinja"
    local_template = Path("/content") / template_name

    if local_template.exists() and local_template.stat().st_size > 0:
        return local_template

    search_roots = []

    try:
        import vllm as vllm_pkg
        vllm_path = Path(vllm_pkg.__file__).resolve()
        search_roots.extend([
            vllm_path.parent,
            vllm_path.parent.parent,
        ])
    except Exception:
        pass

    search_roots.extend([
        Path("/usr/local/lib/python3.12/dist-packages"),
        Path("/usr/local/lib/python3.12/site-packages"),
        Path("/usr/lib/python3.12/site-packages"),
        Path("/content"),
    ])

    for root in search_roots:
        if not root.exists():
            continue

        try:
            matches = sorted(root.rglob(template_name))
        except Exception:
            matches = []

        for match in matches:
            if match.exists() and match.stat().st_size > 0:
                print("Found Gemma 4 chat template:", match)
                return match

    try:
        import requests

        url = (
            "https://raw.githubusercontent.com/vllm-project/vllm/main/"
            "examples/tool_chat_template_gemma4.jinja"
        )
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        local_template.write_text(r.text, encoding="utf-8")

        if local_template.exists() and local_template.stat().st_size > 0:
            print("Downloaded Gemma 4 chat template:", local_template)
            return local_template

    except Exception as e:
        print("Warning: could not download Gemma 4 chat template:", repr(e))

    return None

patch_prometheus_fastapi_instrumentator_routing_again()

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

gemma4_chat_template_path = find_or_download_gemma4_chat_template()

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only QA workload.
    "--language-model-only",
    "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',

    # Non-thinking mode.
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",

    # Force Triton MoE backend to avoid FlashInfer CUTLASS MoE JIT crash on this vLLM nightly.
    "--moe-backend", "triton",

    "--trust-remote-code",
]

if gemma4_chat_template_path is not None:
    cmd.extend(["--chat-template", str(gemma4_chat_template_path)])
else:
    print("Warning: running without explicit Gemma 4 chat template.")

server_env = os.environ.copy()

# Blackwell / CUDA 13 fixes.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server in non-thinking mode.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Prometheus routing patch already applied: /usr/local/lib/python3.12/dist-packages/prometheus_fastapi_instrumentator/routing.py
Command:
vllm serve google/gemma-4-26B-A4B-it --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.92 --language-model-only --limit-mm-per-prompt '{"image": 0, "audio": 0}' --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --moe-backend triton --trust-remote-code --chat-template /content/tool_chat_template_gemma4.jinja

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server in non-thinking mode.
PID: 9042
Log: /content/vllm_gemma4_answer_server.log


In [6]:
# cell 6
# Wait for vLLM server and create OpenAI-compatible client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

def endpoint_ok(url, timeout=10):
    # Return True only if an endpoint responds with HTTP 200.
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code == 200, r
    except Exception:
        return False, None

def log_has_ready_signal(path):
    # Detect server readiness from vLLM/Uvicorn logs.
    text = tail_log(path, n=400)
    ready_signals = [
        "Application startup complete",
        "Uvicorn running on",
    ]
    return any(s in text for s in ready_signals)

def quick_chat_smoke_test(openai_client):
    # Test the real OpenAI-compatible chat endpoint.
    completion = openai_client.chat.completions.create(
        model=SERVER_MODEL_ID or LLM_MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": (
                    "Given the following question, create a final answer in English to the question. "
                    "QUESTION: What is the capital of France? "
                    "ANSWER: [Please provide only the answer and keep the answer less than 6 words. "
                    "If you do not know the answer, write exactly \"I don't know\".]"
                ),
            }
        ],
        max_tokens=64,
        temperature=1.0,
        top_p=0.95,
        presence_penalty=0.0,
        extra_body={
            "top_k": 64,
            "chat_template_kwargs": {
                "enable_thinking": False,
            },
        },
    )

    msg = completion.choices[0].message
    content = getattr(msg, "content", None)

    print("Smoke test content:", content)
    return completion

ready = False
SERVER_MODEL_ID = None

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC
last_smoke_test_time = -999

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=220))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    models_ok, models_response = endpoint_ok(f"{BASE_URL}/models", timeout=10)

    if models_ok:
        model_info = models_response.json()["data"][0]
        SERVER_MODEL_ID = model_info["id"]
        ready = True
        print("vLLM server is ready.")
        print("Model:", SERVER_MODEL_ID)
        print("Max model len:", model_info.get("max_model_len"))
        break

    if log_has_ready_signal(SERVER_LOG_PATH) and elapsed - last_smoke_test_time >= 60:
        last_smoke_test_time = elapsed

        try:
            smoke = quick_chat_smoke_test(client)
            ready = True
            SERVER_MODEL_ID = LLM_MODEL_NAME
            print("vLLM chat endpoint is ready.")
            break
        except Exception as e:
            print("Smoke test failed, continuing to wait:", repr(e))

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed

        health_ok, _ = endpoint_ok(f"http://localhost:{PORT}/health", timeout=5)

        print(f"Waiting... {elapsed}s")
        print("Health endpoint OK:", health_ok)
        print("Models endpoint OK:", models_ok)
        print("Ready signal in log:", log_has_ready_signal(SERVER_LOG_PATH))

        recent = tail_log(SERVER_LOG_PATH, n=20)
        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=260))
    raise RuntimeError("vLLM server did not become ready before timeout.")

# Required standardized sampling configuration.
LLM_SAMPLING_KWARGS = {
    "temperature": 1.0,
    "top_p": 0.95,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 64,
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)
print("Non-thinking extra body:", LLM_EXTRA_BODY)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)

Waiting... 0s
Health endpoint OK: False
Models endpoint OK: False
Ready signal in log: False
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339] 
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339]        █     █     █▄   ▄█
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.22.1rc1.dev511+gc621af169
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339]   █▄█▀ █     █     █     █  model   google/gemma-4-26B-A4B-it
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:339] 
(APIServer pid=9042) INFO 06-14 16:14:49 [api_utils.py:273] non-default args: {'model_tag': 'google/gemma-4-26B-A4B-it', 'chat_template': '/content/tool_chat_template_gemma4.jinja', 'default_chat_template_kwargs': {'enable_thinking': False}, 'host': '0.0.0.0', 'model': 'google/gemma-4-26B-A4B-it', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 1

In [7]:
# cell 7
# Question-only prompt.

QUESTION_ONLY_PROMPT_TEMPLATE = """
Given the following question, create a final answer in English to the question.

QUESTION: {question}

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly "I don't know".]
""".strip()

def clean_text_for_prompt(text):
    # Preserve text content, only normalize Python None.
    if text is None:
        return ""
    return str(text).strip()

def build_answer_prompt(record):
    # Build a question-only prompt.
    # Important: evidence_chunk, supports, and any context fields are intentionally ignored.
    question = clean_text_for_prompt(record.get("question", ""))

    return QUESTION_ONLY_PROMPT_TEMPLATE.format(question=question)

# Preview one prompt from each dataset.
for cfg in DATASET_CONFIGS:
    dataset_name = cfg["name"]
    records = dataset_records[dataset_name]

    print("=" * 100)
    print("Dataset:", dataset_name)

    if records:
        preview_prompt = build_answer_prompt(records[0])
        print(preview_prompt)
        print("\nPrompt preview length in characters:", len(preview_prompt))

Dataset: 2wikimultihopqa
Given the following question, create a final answer in English to the question.

QUESTION: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly "I don't know".]

Prompt preview length in characters: 313
Dataset: hotpotqa
Given the following question, create a final answer in English to the question.

QUESTION: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly "I don't know".]

Prompt preview length in characters: 325


In [8]:
# cell 8
# Response extraction and cleanup helpers.

def strip_code_fence(text):
    # Remove markdown code fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json|text)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Safety cleanup only. In non-thinking mode, these should normally not appear.
    text = text or ""

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")

    # Remove generic special tokens.
    text = re.sub(r"<\|[^>]+?\|>", "", text)

    return text.strip()

def normalize_answer_text(answer):
    # Keep the final answer short and clean.
    answer = strip_think_blocks(answer)
    answer = strip_code_fence(answer)
    answer = re.sub(r"\s+", " ", answer).strip()

    if not answer:
        return "I don't know"

    # Remove accidental labels.
    answer = re.sub(r"^\s*ANSWER\s*:\s*", "", answer, flags=re.IGNORECASE).strip()
    answer = re.sub(r"^\s*FINAL\s*:\s*", "", answer, flags=re.IGNORECASE).strip()
    answer = re.sub(r"^\s*RESPONSE\s*:\s*", "", answer, flags=re.IGNORECASE).strip()

    # Keep only the first non-empty line.
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    if lines:
        answer = lines[0].strip()

    # Remove surrounding quotes.
    answer = answer.strip().strip('"').strip("'").strip()

    if not answer:
        return "I don't know"

    # Normalize common unknown responses.
    unknown_variants = {
        "i do not know",
        "i don't know",
        "unknown",
        "not known",
        "not available",
        "information not available",
        "cannot determine",
        "not enough information",
        "insufficient information",
    }

    if answer.lower().strip(".") in unknown_variants:
        return "I don't know"

    return answer

def extract_chat_message_content(completion):
    # Extract final message content from an OpenAI-compatible chat completion.
    message = completion.choices[0].message

    content = getattr(message, "content", None)

    if content is None and isinstance(message, dict):
        content = message.get("content")

    if content is None:
        content = ""

    return str(content)

print("Response cleanup helpers are ready.")

Response cleanup helpers are ready.


In [9]:
# cell 9
# LLM answer function in non-thinking mode.

def call_llm_answer(prompt, max_retries=3):
    # Call the local vLLM OpenAI-compatible server and return a cleaned final answer.
    last_error = None
    last_raw_content = None
    last_usage = None

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=SERVER_MODEL_ID or LLM_MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                max_tokens=ANSWER_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            raw_content = extract_chat_message_content(completion)
            cleaned = normalize_answer_text(raw_content)

            last_raw_content = raw_content

            try:
                last_usage = completion.usage.model_dump() if completion.usage else None
            except Exception:
                last_usage = None

            if cleaned:
                return {
                    "response": cleaned,
                    "raw_content": raw_content,
                    "usage": last_usage,
                    "attempt": attempt,
                    "retry_reason": None,
                }

            print(f"Warning: empty final answer on attempt {attempt}/{max_retries}. Retrying...")
            time.sleep(2 * attempt)

        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt}/{max_retries}: {repr(e)}")
            time.sleep(2 * attempt)

    if last_error is not None:
        print("Final LLM error:", repr(last_error))

    return {
        "response": "I don't know",
        "raw_content": last_raw_content,
        "usage": last_usage,
        "attempt": "fallback",
        "retry_reason": "failed_after_retries",
        "error": repr(last_error) if last_error is not None else None,
    }

print("Non-thinking LLM function is ready.")

Non-thinking LLM function is ready.


In [10]:
# cell 10
# Test one question-only LLM call before running the full datasets.

test_dataset = "hotpotqa"
test_record = dataset_records[test_dataset][0]

test_prompt = build_answer_prompt(test_record)

print("Test dataset:", test_dataset)
print("Test question:", test_record["question"])
print("Test GT answer:", test_record["answer"])
print("Note: GT is printed only for human checking and is not sent to the LLM.")
print("\nPrompt sent to LLM:")
print(test_prompt)

test_result = call_llm_answer(test_prompt)

print("\nTest LLM response:", test_result["response"])
print("Attempt:", test_result.get("attempt"))

if test_result.get("raw_content"):
    print("\nRaw content preview:")
    print(str(test_result["raw_content"])[:1000])

Test dataset: hotpotqa
Test question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Test GT answer: Bedknobs and Broomsticks
Note: GT is printed only for human checking and is not sent to the LLM.

Prompt sent to LLM:
Given the following question, create a final answer in English to the question.

QUESTION: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?

ANSWER: [Please provide only the answer and keep the answer less than 6 words. If you do not know the answer, write exactly "I don't know".]

Test LLM response: Bedknobs and Broomsticks
Attempt: 1

Raw content preview:
Bedknobs and Broomsticks


In [11]:
# cell 11
# Output I/O and resume helpers.

def atomic_write_json(path, data):
    # Atomic JSON write.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_outputs(path):
    # Load existing answers for resume.
    path = Path(path)

    if not path.exists():
        return []

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return []

    if not isinstance(data, list):
        return []

    cleaned = []

    for rec in data:
        if not isinstance(rec, dict):
            continue

        # Keep only the required output keys.
        cleaned.append({
            "type": rec.get("type"),
            "question": rec.get("question"),
            "gt": rec.get("gt"),
            "response": rec.get("response"),
        })

    return cleaned

def save_outputs(path, outputs):
    # Save outputs in dataset order.
    atomic_write_json(path, outputs)

def make_output_record(record, llm_response):
    # Build the final record with only the requested keys.
    return {
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
        "response": llm_response,
    }

def make_error_output_record(record):
    # Keep the required schema even if an API/runtime error happens.
    return make_output_record(record, "I don't know")

def update_output_at_index(outputs, index, output_record):
    # Update or append one output record while preserving dataset order.
    if index < len(outputs):
        outputs[index] = output_record
    elif index == len(outputs):
        outputs.append(output_record)
    else:
        raise RuntimeError(
            f"Cannot write index {index}; output currently has only {len(outputs)} records. "
            "If using ANSWER_START_INDEX > 0, the previous records must already exist in the output file."
        )

for cfg in DATASET_CONFIGS:
    existing = load_existing_outputs(cfg["drive_output_path"])

    print("=" * 100)
    print("Dataset:", cfg["name"])
    print("Existing output records:", len(existing))
    print("Output path:", cfg["drive_output_path"])

Dataset: 2wikimultihopqa
Existing output records: 20
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json
Dataset: hotpotqa
Existing output records: 0
Output path: /content/drive/MyDrive/final_project/LLM/hotpotqa_gemma4_answers.json


In [12]:
# cell 12
# Run answer generation for both datasets separately.

def run_dataset_answer_generation(cfg, records):
    # Generate answers for one dataset and save it to its own output JSON file.
    dataset_name = cfg["name"]
    output_path = cfg["drive_output_path"]

    outputs = load_existing_outputs(output_path)

    start_idx = int(ANSWER_START_INDEX)
    end_idx = len(records) if ANSWER_END_INDEX is None else int(ANSWER_END_INDEX)
    end_idx = min(end_idx, len(records))

    if start_idx > len(outputs):
        raise RuntimeError(
            f"Dataset {dataset_name}: ANSWER_START_INDEX={start_idx}, "
            f"but existing output has only {len(outputs)} records. "
            "Run from index 0 first or keep a complete previous output file."
        )

    run_indices = range(start_idx, end_idx)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Total evidence records:", len(records))
    print("Run range:", start_idx, "to", end_idx)
    print("Existing outputs:", len(outputs))
    print("Output:", output_path)

    progress = tqdm(run_indices, desc=f"Generating {dataset_name}", dynamic_ncols=True)

    for step, idx in enumerate(progress, start=1):
        record = records[idx]

        # Skip completed records.
        if idx < len(outputs):
            old = outputs[idx]
            if isinstance(old, dict) and old.get("response"):
                progress.set_postfix({
                    "idx": idx,
                    "status": "skipped",
                    "saved": len(outputs),
                })
                continue

        try:
            prompt = build_answer_prompt(record)
            llm_result = call_llm_answer(prompt)

            output_record = make_output_record(record, llm_result["response"])
            status = "ok"

        except Exception as e:
            print(f"\nError in dataset={dataset_name}, index={idx}")
            print("Question:", record.get("question"))
            print("Error:", repr(e))
            print(traceback.format_exc())

            output_record = make_error_output_record(record)
            status = "error"

        update_output_at_index(outputs, idx, output_record)

        if step % SAVE_EVERY_N == 0:
            save_outputs(output_path, outputs)

        if step % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        progress.set_postfix({
            "idx": idx,
            "status": status,
            "saved": len(outputs),
        })

    save_outputs(output_path, outputs)

    print("Finished dataset:", dataset_name)
    print("Saved records:", len(outputs))
    print("Output file:", output_path)

for cfg in DATASET_CONFIGS:
    records = dataset_records[cfg["name"]]

    # Each dataset is processed independently.
    # No records, prompts, or outputs are mixed between datasets.
    run_dataset_answer_generation(cfg, records)

print("All datasets finished.")

Dataset: 2wikimultihopqa
Total evidence records: 1000
Run range: 0 to 1000
Existing outputs: 20
Output: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json


Generating 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: 2wikimultihopqa
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json
Dataset: hotpotqa
Total evidence records: 1000
Run range: 0 to 1000
Existing outputs: 0
Output: /content/drive/MyDrive/final_project/LLM/hotpotqa_gemma4_answers.json


Generating hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

Finished dataset: hotpotqa
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/LLM/hotpotqa_gemma4_answers.json
All datasets finished.


In [13]:
# cell 13
# Inspect saved answers for both datasets.

for cfg in DATASET_CONFIGS:
    output_path = cfg["drive_output_path"]

    print("=" * 100)
    print("Dataset:", cfg["name"])
    print("Output path:", output_path)

    with open(output_path, "r", encoding="utf-8") as f:
        saved_answers = json.load(f)

    print("Saved answers:", len(saved_answers))

    for rec in saved_answers[:5]:
        print("-" * 100)
        print("type:", rec.get("type"))
        print("question:", rec.get("question"))
        print("gt:", rec.get("gt"))
        print("response:", rec.get("response"))

Dataset: 2wikimultihopqa
Output path: /content/drive/MyDrive/final_project/LLM/2wikimultihopqa_gemma4_answers.json
Saved answers: 1000
----------------------------------------------------------------------------------------------------
type: bridge_comparison
question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
gt: Kamakalawa
response: I don't know
----------------------------------------------------------------------------------------------------
type: compositional
question: Where did Prince Gustav Of Thurn And Taxis (1848–1914)'s mother die?
gt: Meran
response: Regensburg
----------------------------------------------------------------------------------------------------
type: bridge_comparison
question: Which film has the director died later, A Light Woman or Our Mother'S House?
gt: Our Mother'S House
response: Our Mother's House
----------------------------------------------------------------------------------------------------
type: bridge_co